In [1]:
import pandas as pd
import sqlite3
import numpy as np
import time
import os
import requests
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle, Rectangle, Arc
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from nba_api.stats.static import players
from nba_api.stats.endpoints import commonplayerinfo
from nba_api.stats.endpoints import shotchartdetail
from nba_api.stats.static import teams


In [2]:
conn = sqlite3.connect('nba_centers_info.db')


In [3]:
#Retrieve the shot location dataframe
shot_loc_df = pd.read_sql("SELECT * FROM ShotLocs", conn)
shot_loc_df

,PLAYER_ID,PLAYER_NAME,TEAM_NAME,GAME_DATE,PERIOD,EVENT_TYPE,SHOT_TYPE,SHOT_DISTANCE,LOC_X,LOC_Y
0,2561,David West,Indiana Pacers,20131029,1,Missed Shot,2PT Field Goal,5.0,-38.0,45.0
1,202331,Paul George,Indiana Pacers,20131029,1,Made Shot,2PT Field Goal,19.0,105.0,164.0
2,201167,Arron Afflalo,Orlando Magic,20131029,1,Missed Shot,3PT Field Goal,27.0,51.0,266.0
3,2561,David West,Indiana Pacers,20131029,1,Missed Shot,2PT Field Goal,2.0,28.0,-5.0
4,202362,Lance Stephenson,Indiana Pacers,20131029,1,Made Shot,3PT Field Goal,26.0,15.0,260.0
...,...,...,...,...,...,...,...,...,...,...
2510333,1630578,Alperen Sengun,Houston Rockets,20241214,4,Missed Shot,3PT Field Goal,25.0,-29.0,257.0
2510334,1627832,Fred VanVleet,Houston Rockets,20241214,4,Missed Shot,3PT Field Goal,22.0,-228.0,-17.0
2510335,1629652,Luguentz Dort,Oklahoma City Thunder,20241214,4,Made Shot,3PT Field Goal,24.0,186.0,167.0
2510336,1630224,Jalen Green,Houston Rockets,20241214,4,Made Shot,2PT Field Goal,2.0,12.0,16.0


In [4]:
centers_df = pd.read_sql("SELECT * FROM Centers", conn)
centers_df

,PERSON_ID,DISPLAY_FIRST_LAST,POSITION,HEIGHT,TEAM_NAME
0,203500,Steven Adams,Center,6-11,Rockets
1,1628386,Jarrett Allen,Center,6-9,Cavaliers
2,1629028,Deandre Ayton,Center,7-0,Lakers
3,202687,Bismack Biyombo,Center,6-8,Spurs
4,203991,Clint Capela,Center,6-10,Rockets
...,...,...,...,...,...
696,1917,Wang Zhi-zhi,Center,7-1,
697,678,George Zidek,Center,7-0,
698,1627757,Stephen Zimmerman,Center,7-0,Magic
699,1627790,Ante Zizic,Center,6-10,Cavaliers


In [5]:
#Inner join the centers and shot location tables, then output as a Pandas dataframe
center_loc_df = pd.read_sql('''SELECT 
            sl.PLAYER_NAME,
            c.POSITION,
            sl.GAME_DATE,
            sl.TEAM_NAME,
            sl.EVENT_TYPE,
            sl.SHOT_TYPE,
            sl.SHOT_DISTANCE,
            sl.LOC_X,
            sl.LOC_Y
            FROM ShotLocs as sl
            INNER JOIN Centers as c
            ON sl.PLAYER_ID = c.PERSON_ID''', conn)
center_loc_df

,PLAYER_NAME,POSITION,GAME_DATE,TEAM_NAME,EVENT_TYPE,SHOT_TYPE,SHOT_DISTANCE,LOC_X,LOC_Y
0,Roy Hibbert,Center,20131029,Indiana Pacers,Missed Shot,2PT Field Goal,1.0,13.0,9.0
1,Roy Hibbert,Center,20131029,Indiana Pacers,Made Shot,2PT Field Goal,0.0,4.0,0.0
2,Roy Hibbert,Center,20131029,Indiana Pacers,Missed Shot,2PT Field Goal,9.0,-97.0,15.0
3,Nikola Vučević,Center,20131029,Orlando Magic,Made Shot,2PT Field Goal,4.0,-48.0,7.0
4,Nikola Vučević,Center,20131029,Orlando Magic,Missed Shot,2PT Field Goal,1.0,12.0,0.0
...,...,...,...,...,...,...,...,...,...
201033,Alperen Sengun,Center,20241214,Houston Rockets,Made Shot,2PT Field Goal,4.0,-17.0,41.0
201034,Alperen Sengun,Center,20241214,Houston Rockets,Made Shot,3PT Field Goal,26.0,146.0,220.0
201035,Alperen Sengun,Center,20241214,Houston Rockets,Made Shot,2PT Field Goal,11.0,64.0,101.0
201036,Alperen Sengun,Center,20241214,Houston Rockets,Made Shot,2PT Field Goal,1.0,10.0,0.0


In [8]:
#Create variables for the game year in both dataframes.
center_loc_df['YEAR'] = center_loc_df['GAME_DATE'].str[:-4] 
shot_loc_df['YEAR'] = shot_loc_df['GAME_DATE'].str[:-4] 

#Create a function to identify zones
def assign_zone(row):
    dist = row['SHOT_DISTANCE']
    x, y = row['LOC_X'], row['LOC_Y']
    
    if dist <= 4:
        return 'Restricted Area'
    elif dist <= 8:
        return 'Paint'
    elif dist < 22:
        return 'Mid-Range'
    elif abs(x) >= 220 and y <= 92.5:
        return 'Corner Three'
    elif dist >= 22:
        return 'Above-the-Break Three'
    else:
        return 'Mid-Range'

#apply assign_zone function to both dataframes to create a shot zone variable
center_loc_df['SHOT_ZONE'] = center_loc_df.apply(assign_zone, axis=1)
shot_loc_df['SHOT_ZONE'] = shot_loc_df.apply(assign_zone, axis=1)



In [9]:
#Save center_loc_df and shot_loc_df locally
center_loc_df.to_csv('center_loc.csv', index = False)
shot_loc_df.to_csv('shot_loc.csv', index = False)

In [10]:
#We are now done with the database, so we can close it. 
conn.close()